In [1]:
import streamlit as st
import pandas as pd
import re

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    pipeline
)

# =====================================================
# PAGE CONFIG
# =====================================================

st.set_page_config(
    page_title="Telecom AI Brand Intelligence System",
    layout="wide"
)

st.title("📡 Telecom AI Brand Intelligence System")

# =====================================================
# LOAD DISTILBERT MODEL
# =====================================================

MODEL_PATH = "models/distilbert_model"

@st.cache_resource
def load_model():
    tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_PATH)

    model = DistilBertForSequenceClassification.from_pretrained(
        MODEL_PATH
    )

    sentiment_pipeline = pipeline(
        "sentiment-analysis",
        model=model,
        tokenizer=tokenizer
    )

    return sentiment_pipeline

sentiment_pipeline = load_model()

# =====================================================
# TEXT CLEANING
# =====================================================

def clean_text(text):

    text = text.lower()

    text = re.sub(r"http\S+", "", text)

    text = re.sub(r"@\w+", "", text)

    text = re.sub(r"[^a-zA-Z\s]", "", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()

# =====================================================
# SENTIMENT PREDICTION
# =====================================================

def get_sentiment(text):

    result = sentiment_pipeline(text[:512])[0]

    label = result["label"]

    if label in ["LABEL_1", "POSITIVE"]:
        return "Positive"

    return "Negative"

# =====================================================
# CATEGORY DETECTION
# =====================================================

def detect_category(text):

    text = text.lower()

    if any(word in text for word in [
        "internet",
        "wifi",
        "broadband"
    ]):
        return "Broadband Service"

    elif any(word in text for word in [
        "bill",
        "payment",
        "charge"
    ]):
        return "Billing & Payments"

    elif any(word in text for word in [
        "support",
        "help"
    ]):
        return "Customer Support"

    elif any(word in text for word in [
        "sim",
        "activation"
    ]):
        return "Service Activation"

    elif any(word in text for word in [
        "network",
        "signal",
        "coverage"
    ]):
        return "Mobile Network"

    else:
        return "General Complaint"

# =====================================================
# USER INPUT
# =====================================================

st.header("📝 Customer Feedback Analysis")

feedback = st.text_area(
    "Enter Customer Feedback",
    height=150
)

if st.button("Analyze Feedback"):

    if feedback.strip() == "":
        st.warning("Please enter feedback.")

    else:

        cleaned = clean_text(feedback)

        sentiment = get_sentiment(cleaned)

        category = detect_category(cleaned)

        col1, col2 = st.columns(2)

        with col1:
            st.subheader("📊 Sentiment")
            st.success(sentiment)

        with col2:
            st.subheader("🏷️ Category")
            st.info(category)

        st.subheader("🤖 AI Response")

        response = f"""
        We understand your concern regarding {category}.

        Our analysis detected a {sentiment.lower()} customer sentiment.

        Our support team will review the issue and provide assistance as soon as possible.

        Thank you for contacting ZENDS Communications.
        """

        st.write(response)

# =====================================================
# DASHBOARD
# =====================================================

st.markdown("---")

st.header("📈 Telecom Brand Analyst Dashboard")

try:

    df = pd.read_csv("telecom_ai_dataset.csv")

    col1, col2, col3 = st.columns(3)

    col1.metric(
        "Total Feedback",
        len(df)
    )

    if "Sentiment" in df.columns:

        positive = len(
            df[df["Sentiment"] == "Positive"]
        )

        negative = len(
            df[df["Sentiment"] == "Negative"]
        )

        col2.metric(
            "Positive Feedback",
            positive
        )

        col3.metric(
            "Negative Feedback",
            negative
        )

        st.subheader("Sentiment Distribution")

        st.bar_chart(
            df["Sentiment"].value_counts()
        )

    if "Service_Category" in df.columns:

        st.subheader(
            "Service Category Distribution"
        )

        st.bar_chart(
            df["Service_Category"].value_counts()
        )

    st.subheader("Dataset Preview")

    st.dataframe(df.head(20))

except Exception as e:

    st.error(
        f"Could not load dataset: {e}"
    )

# =====================================================
# FOOTER
# =====================================================

st.markdown("---")

st.caption(
    "🚀 Built using DistilBERT + Streamlit"
)

ModuleNotFoundError: No module named 'cgi'

In [3]:
pip show transformers

Name: transformersNote: you may need to restart the kernel to use updated packages.

Version: 5.8.0
Summary: Transformers: the model-definition framework for state-of-the-art machine learning models in text, vision, audio, and multimodal models, for both inference and training.
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: C:\Users\admin\AppData\Local\Programs\Python\Python313\Lib\site-packages
Requires: huggingface-hub, numpy, packaging, pyyaml, regex, safetensors, tokenizers, tqdm, typer
Required-by: sentence-transformers
